In [10]:
import os 
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy.stats import t as t_dist

# ---------------------------
# 1. Read cryptocurrency data
# ---------------------------
crypto_types = {
    "BCH": "brown", "BTC": "brown", "ETC": "brown", "ETH": "brown",
    "LTC": "brown", "XMR": "brown", "ADA": "green", "ATOM": "green",
    "MATIC": "green", "TRX": "green", "XRP": "green", "XLM": "green"
}
directory = r"D:\桌面\绿色货币\数据总库\加密货币数据\coincodex"
dfs = []
for f in os.listdir(directory):
    if f.endswith(".csv") and f.split("_")[0] in crypto_types:
        try:
            temp = pd.read_csv(os.path.join(directory, f), parse_dates=["End"])
            temp.drop(columns=["Start"], errors="ignore", inplace=True)
            ticker = f.split("_")[0]
            temp["source"] = ticker
            temp["Type"] = crypto_types[ticker]
            dfs.append(temp)
        except Exception as e:
            print(f"Error reading {f}: {e}")

data = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# ---------------------------
# 2. Process data format
# ---------------------------
df = data.copy()
df["End"] = pd.to_datetime(df["End"])
df = df[df["End"] >= "2019-06-30"]
df = df.sort_values(["source", "End"]).reset_index(drop=True)

# ---------------------------
# 3. Calculate log returns (rtn) and construct next period returns (rtn_next)
# ---------------------------
df["rtn"] = df.groupby("source")["Close"].transform(lambda x: np.log(x).diff())
# 生成下一期收益：对于每个资产，t=0时的收益对应下一期（t+1）的收益
df["rtn_next"] = df.groupby("source")["rtn"].shift(-1)
df = df.dropna()

# ---------------------------
# 4. Construct the three factors using information at time t
# ---------------------------
# 4.1 CMKT (market factor) using current period returns (rtn)
CMKT = df.groupby("End").apply(lambda g: np.sum(g["rtn"] * g["Market Cap"]) / np.sum(g["Market Cap"]))
CMKT.name = "CMKT"

# 4.2 CSMB (size factor)
df_size = df.groupby("End", group_keys=False).apply(
    lambda sub: sub.assign(Size_Group=pd.qcut(sub["Market Cap"], [0, 0.3, 0.7, 1.0], labels=["Small", "Middle", "Big"]))
).reset_index(drop=True)

CSMB = (df_size[df_size["Size_Group"] == "Small"].groupby("End")["rtn"].mean() -
        df_size[df_size["Size_Group"] == "Big"].groupby("End")["rtn"].mean())
CSMB.name = "CSMB"

# 计算三周期动量（采用三周动量），仍使用当前期收益构造
df_size["momentum"] = df_size.groupby("source")["rtn"].transform(
    lambda x: (1 + x).rolling(window=3, min_periods=3).apply(np.prod, raw=True) - 1)
df_size = df_size.dropna()

# 只保留 Small 和 Big 两个市场规模组的数据
df_mom = df_size[df_size["Size_Group"].isin(["Small", "Big"])].copy()

# 对于每个日期和市场规模组，将动量分为三个分组：Low, Middle, High
def assign_momentum_group(sub):
    sub = sub.copy()
    sub["Momentum_Group"] = pd.qcut(sub["momentum"], q=[0, 0.3, 0.7, 1.0],
                                    labels=["Low", "Middle", "High"])
    return sub

df_mom = df_mom.groupby(["End", "Size_Group"], group_keys=False).apply(assign_momentum_group)

# 定义函数计算每个日期的动量因子 CMOM
def compute_cmom(group):
    result = {}
    for size in ["Small", "Big"]:
        sub = group[group["Size_Group"] == size]
        if not sub.empty:
            high_mean = sub[sub["Momentum_Group"] == "High"]["rtn"].mean()
            low_mean  = sub[sub["Momentum_Group"] == "Low"]["rtn"].mean()
            result[size] = (high_mean, low_mean)
        else:
            result[size] = (np.nan, np.nan)
    if (pd.notnull(result["Small"][0]) and pd.notnull(result["Small"][1]) and 
        pd.notnull(result["Big"][0]) and pd.notnull(result["Big"][1])):
        cmom = 0.5 * (result["Small"][0] + result["Big"][0]) - 0.5 * (result["Small"][1] + result["Big"][1])
    else:
        cmom = np.nan
    return cmom

CMOM = df_mom.groupby("End").apply(compute_cmom)
CMOM.name = "CMOM"

# ---------------------------
# 5. Read 1-month US Treasury yield data FRED and calculate excess returns
# ---------------------------
rf_file_path = "D:\\桌面\\绿色货币\\数据总库\\DGS1MO.csv"  # Update the path as needed
rf_data = pd.read_csv(rf_file_path)
rf_data["Date"] = pd.to_datetime(rf_data["Date"], format="%Y/%m/%d")
rf_data["DGS1MO"] = rf_data["DGS1MO"] / 100  # Convert to decimals
rf_data = rf_data.rename(columns={"Date": "End", "DGS1MO": "Rf"})
rf_data = rf_data.ffill()

# Merge risk-free rate with df based on current period End
df = df.sort_values("End")
df = df.merge(rf_data, on="End", how="left")
df["Rf"] = df["Rf"].fillna(method='ffill')

# 为下一期构造风险溢价：对每个资产，取下一期的 Rf
df["Rf_next"] = df.groupby("source")["Rf"].shift(-1)
# 构造下一期超额收益：rp_next = rtn_next - Rf_next
df["rp_next"] = df["rtn_next"] - df["Rf_next"]

# ---------------------------
# 6. Prepare factor DataFrame (using current period t factors)
# ---------------------------
test_factors = pd.concat([CMKT, CSMB, CMOM], axis=1).dropna().reset_index()
# 使用当前 t 时点的风险溢价来调整 CMKT（构造超额市场因子）
test_factors["prev_date"] = test_factors["End"] - pd.Timedelta(days=1)

test_factors = pd.merge_asof(
    test_factors.sort_values("prev_date"),
    rf_data.sort_values("End"),
    left_on="prev_date",
    right_on="End",
    direction="backward"
)
test_factors.drop(columns=["prev_date", "End_y"], inplace=True)
test_factors.rename(columns={"End_x": "End"}, inplace=True)
test_factors["CMKT"] = test_factors["CMKT"] - test_factors["Rf"]

# ---------------------------
# 7. Perform regression analysis using next period excess returns (rp_next)
# ---------------------------
# 这里合并时用的是当前时点的因子，与各资产下一期的超额收益对应
df_reg = pd.merge(df, test_factors, on="End", how="inner")

# 使用下一期超额收益 rp_next 作为被解释变量进行回归
green_model = smf.ols("rp_next ~ CMKT + CSMB + CMOM", data=df_reg[df_reg["Type"] == "green"]).fit()
brown_model = smf.ols("rp_next ~ CMKT + CSMB + CMOM", data=df_reg[df_reg["Type"] == "brown"]).fit()

print(green_model.summary())
print(brown_model.summary())

alpha_green = green_model.params["Intercept"]
se_green = green_model.bse["Intercept"]
alpha_brown = brown_model.params["Intercept"]
se_brown = brown_model.bse["Intercept"]

t_stat = (alpha_green - alpha_brown) / np.sqrt(se_green**2 + se_brown**2)
df_eff = min(green_model.df_resid, brown_model.df_resid)
p_val = 2 * (1 - t_dist.cdf(abs(t_stat), df=df_eff))

print("\nAlpha difference test between Green and Brown groups:")
print(f"Green alpha: {alpha_green:.4f} (SE={se_green:.4f})")
print(f"Brown alpha: {alpha_brown:.4f} (SE={se_brown:.4f})")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.4f}")


C:\Users\why97\AppData\Local\Temp\ipykernel_30364\1515039808.py:52: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CMKT = df.groupby("End").apply(lambda g: np.sum(g["rtn"] * g["Market Cap"]) / np.sum(g["Market Cap"]))
C:\Users\why97\AppData\Local\Temp\ipykernel_30364\1515039808.py:56: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_size = df.groupby("End", group_keys=False).apply(
C:\Users\why97\AppData\Loc

                            OLS Regression Results                            
Dep. Variable:                rp_next   R-squared:                       0.033
Model:                            OLS   Adj. R-squared:                  0.033
Method:                 Least Squares   F-statistic:                     136.2
Date:                Fri, 14 Mar 2025   Prob (F-statistic):           9.27e-87
Time:                        10:38:00   Log-Likelihood:                 16903.
No. Observations:               12042   AIC:                        -3.380e+04
Df Residuals:                   12038   BIC:                        -3.377e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0223      0.001    -30.264      0.0

C:\Users\why97\AppData\Local\Temp\ipykernel_30364\1515039808.py:99: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CMOM = df_mom.groupby("End").apply(compute_cmom)
C:\Users\why97\AppData\Local\Temp\ipykernel_30364\1515039808.py:115: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["Rf"] = df["Rf"].fillna(method='ffill')


week


In [11]:
import os  
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy.stats import t as t_dist

# ---------------------------
# 1. Read data 
# ---------------------------
crypto_types = {
    "BCH": "brown", "BTC": "brown", "ETC": "brown", "ETH": "brown",
    "LTC": "brown", "XMR": "brown", "ADA": "green", "ATOM": "green",
    "MATIC": "green", "TRX": "green", "XRP": "green", "XLM": "green",
    "DOGE": "brown"
}
directory = r"D:\桌面\绿色货币\数据总库\加密货币数据\coincodex"
dfs = []
for f in os.listdir(directory):
    if f.endswith(".csv") and f.split("_")[0] in crypto_types:
        try:
            temp = pd.read_csv(os.path.join(directory, f), parse_dates=["End"])
            temp.drop(columns=["Start"], errors="ignore", inplace=True)
            ticker = f.split("_")[0]
            temp["source"] = ticker
            temp["Type"] = crypto_types[ticker]
            dfs.append(temp)
        except Exception as e:
            print(f"Error reading {f}: {e}")

data = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# ---------------------------
# 2. Process data format and aggregate to weekly frequency
# ---------------------------
df = data.copy()
df["End"] = pd.to_datetime(df["End"])
df = df[df["End"] >= "2019-06-30"]
df = df.sort_values(["source", "End"]).reset_index(drop=True)

# 聚合为周频数据：以每个加密货币每周最后一个交易日的数据为准
df.set_index("End", inplace=True)
# 使用 W-FRI 表示以每周星期五作为汇总日（可根据需要调整）
df_weekly = df.groupby("source").resample("W-FRI").agg({
    "Close": "last",
    "Market Cap": "last",
    "Type": "first"  # "source" 已作为 group key，无需重复处理
}).reset_index()
df = df_weekly.copy()

# ---------------------------
# 3. Calculate log returns (rtn) and construct next period returns
# ---------------------------
df["rtn"] = df.groupby("source")["Close"].transform(lambda x: np.log(x).diff())
# 对每个加密货币，下一期的收益（t+1）由 shift(-1) 得到
df["rtn_next"] = df.groupby("source")["rtn"].shift(-1)
df = df.dropna()

# ---------------------------
# 4. Construct the three factors using information at time t
# ---------------------------
# 4.1 CMKT (market factor)
CMKT = df.groupby("End").apply(lambda g: np.sum(g["rtn"] * g["Market Cap"]) / np.sum(g["Market Cap"]))
CMKT.name = "CMKT"

# 4.2 CSMB (size factor)
df_size = df.groupby("End", group_keys=False).apply(
    lambda sub: sub.assign(Size_Group=pd.qcut(sub["Market Cap"], [0, 0.3, 0.7, 1.0], labels=["Small", "Middle", "Big"]))
).reset_index(drop=True)
CSMB = (df_size[df_size["Size_Group"] == "Small"].groupby("End")["rtn"].mean() -
        df_size[df_size["Size_Group"] == "Big"].groupby("End")["rtn"].mean())
CSMB.name = "CSMB"

# 计算三周期动量（采用三周动量），仍用当前期收益构造
df_size["momentum"] = df_size.groupby("source")["rtn"].transform(
    lambda x: (1 + x).rolling(window=3, min_periods=3).apply(np.prod, raw=True) - 1)
df_size = df_size.dropna()

# 只保留 Small 和 Big 两个市场规模组的数据，用于构造动量因子
df_mom = df_size[df_size["Size_Group"].isin(["Small", "Big"])].copy()
def assign_momentum_group(sub):
    sub = sub.copy()
    sub["Momentum_Group"] = pd.qcut(sub["momentum"], q=[0, 0.3, 0.7, 1.0],
                                    labels=["Low", "Middle", "High"])
    return sub
df_mom = df_mom.groupby(["End", "Size_Group"], group_keys=False).apply(assign_momentum_group)
def compute_cmom(group):
    result = {}
    for size in ["Small", "Big"]:
        sub = group[group["Size_Group"] == size]
        if not sub.empty:
            high_mean = sub[sub["Momentum_Group"] == "High"]["rtn"].mean()
            low_mean  = sub[sub["Momentum_Group"] == "Low"]["rtn"].mean()
            result[size] = (high_mean, low_mean)
        else:
            result[size] = (np.nan, np.nan)
    if (pd.notnull(result["Small"][0]) and pd.notnull(result["Small"][1]) and 
        pd.notnull(result["Big"][0]) and pd.notnull(result["Big"][1])):
        cmom = 0.5 * (result["Small"][0] + result["Big"][0]) - 0.5 * (result["Small"][1] + result["Big"][1])
    else:
        cmom = np.nan
    return cmom
CMOM = df_mom.groupby("End").apply(compute_cmom)
CMOM.name = "CMOM"

# ---------------------------
# 5. Read 1-month US Treasury yield data FRED and calculate excess returns
# ---------------------------
rf_file_path = "D:\\桌面\\绿色货币\\数据总库\\DGS1MO.csv"  # Update the path as needed
rf_data = pd.read_csv(rf_file_path)
rf_data["Date"] = pd.to_datetime(rf_data["Date"], format="%Y/%m/%d")
rf_data["DGS1MO"] = rf_data["DGS1MO"] / 100  # Convert to decimals
rf_data = rf_data.rename(columns={"Date": "End", "DGS1MO": "Rf"})
rf_data = rf_data.ffill()

# 将风险溢价数据也聚合为周频数据（以每周最后一个交易日为准）
rf_data.set_index("End", inplace=True)
rf_weekly = rf_data.resample("W-FRI").last().reset_index()
rf_data = rf_weekly.copy()

# 将风险溢价数据与加密货币数据对齐
df = df.sort_values("End")
df = df.merge(rf_data, on="End", how="left")
df["Rf"] = df["Rf"].fillna(method='ffill')

# 为下一期风险溢价构造滞后项（下一期，即 shift(-1)）
df["Rf_next"] = df.groupby("source")["Rf"].shift(-1)
# 构造下一期超额收益：rp_next = rtn_next - Rf_next
df["rp_next"] = df["rtn_next"] - df["Rf_next"]

# ---------------------------
# 6. Prepare factor DataFrame (using current period t factors)
# ---------------------------
test_factors = pd.concat([CMKT, CSMB, CMOM], axis=1).dropna().reset_index()
test_factors["prev_date"] = test_factors["End"] - pd.Timedelta(days=1)
test_factors = pd.merge_asof(
    test_factors.sort_values("prev_date"),
    rf_data.sort_values("End"),
    left_on="prev_date",
    right_on="End",
    direction="backward"
)
test_factors.drop(columns=["prev_date", "End_y"], inplace=True)
test_factors.rename(columns={"End_x": "End"}, inplace=True)
test_factors["CMKT"] = test_factors["CMKT"] - test_factors["Rf"]

# ---------------------------
# 7. Perform regression analysis using next period excess returns (rp_next)
# ---------------------------
df_reg = pd.merge(df, test_factors, on="End", how="inner")
# 用下一期超额收益 rp_next 作为被解释变量进行回归
green_model = smf.ols("rp_next ~ CMKT + CSMB + CMOM", data=df_reg[df_reg["Type"] == "green"]).fit()
brown_model = smf.ols("rp_next ~ CMKT + CSMB + CMOM", data=df_reg[df_reg["Type"] == "brown"]).fit()

print(green_model.summary())
print(brown_model.summary())

alpha_green = green_model.params["Intercept"]
se_green = green_model.bse["Intercept"]
alpha_brown = brown_model.params["Intercept"]
se_brown = brown_model.bse["Intercept"]

t_stat = (alpha_green - alpha_brown) / np.sqrt(se_green**2 + se_brown**2)
df_eff = min(green_model.df_resid, brown_model.df_resid)
p_val = 2 * (1 - t_dist.cdf(abs(t_stat), df=df_eff))

print("\nAlpha difference test between Green and Brown groups:")
print(f"Green alpha: {alpha_green:.4f} (SE={se_green:.4f})")
print(f"Brown alpha: {alpha_brown:.4f} (SE={se_brown:.4f})")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.4f}")


C:\Users\why97\AppData\Local\Temp\ipykernel_30364\2726132209.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CMKT = df.groupby("End").apply(lambda g: np.sum(g["rtn"] * g["Market Cap"]) / np.sum(g["Market Cap"]))
C:\Users\why97\AppData\Local\Temp\ipykernel_30364\2726132209.py:66: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_size = df.groupby("End", group_keys=False).apply(
C:\Users\why97\AppData\Loc

                            OLS Regression Results                            
Dep. Variable:                rp_next   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.019
Method:                 Least Squares   F-statistic:                     12.13
Date:                Fri, 14 Mar 2025   Prob (F-statistic):           7.45e-08
Time:                        10:38:02   Log-Likelihood:                 809.72
No. Observations:                1698   AIC:                            -1611.
Df Residuals:                    1694   BIC:                            -1590.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0352      0.005     -7.109      0.0

C:\Users\why97\AppData\Local\Temp\ipykernel_30364\2726132209.py:102: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CMOM = df_mom.groupby("End").apply(compute_cmom)
C:\Users\why97\AppData\Local\Temp\ipykernel_30364\2726132209.py:123: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["Rf"] = df["Rf"].fillna(method='ffill')
